# GraphRAG

## Import packages

In [1]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import time
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  summac_zs_metric,
  summac_conv_metric,
)

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


## Disable warnings

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook.

## Import packages

In [3]:
env_variables = [
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
  'OPENROUTER_API_KEY',
  'CHROMA_API_KEY',
  'CHROMA_TENANT',
  'CHROMA_DATABASE',
  'CHROMA_COLLECTION_NAME',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [4]:
app = NeuroRAG(debug=True, use_flare=False)
app.compile()

## Evaluate RAG

### Load QA dataset

In [5]:
mediqa_df = pd.read_csv('../datasets/pubmed_summary_qa.csv')[:20]
mediqa_df

,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...
5,How does the auditory system respond to differ...,The auditory system's response to sound varies...
6,What is tonotopic organization in the auditory...,Tonotopic organization refers to the mapping o...
7,What brain regions are involved in language pr...,Language processing involves areas in the pref...
8,How does bilingualism affect language processi...,Bilingualism is associated with overlapping ac...
9,What is functional magnetic resonance imaging ...,Functional magnetic resonance imaging (fMRI) i...


### Load cached RAGs responses

In [6]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-neurorag-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache.keys())

1

In [7]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []
generation_times = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache[CACHE_KEY]:
    start_time = time.perf_counter()
    cache[CACHE_KEY][question] = app.invoke(question)['generation']
    elapsed = time.perf_counter() - start_time
    generation_times.append(elapsed)

  predicted_answers.append(cache[CACHE_KEY][question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

if generation_times:
  print(f'Generation times (n={len(generation_times)}):')
  print(f'  Mean:   {sum(generation_times) / len(generation_times):.2f}s')
  print(f'  Median: {sorted(generation_times)[len(generation_times) // 2]:.2f}s')
  print(f'  Min:    {min(generation_times):.2f}s')
  print(f'  Max:    {max(generation_times):.2f}s')
  print(f'  Total:  {sum(generation_times):.2f}s')
else:
  print('All answers loaded from cache, no generation times recorded.')

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
bleu_score = bleu_metric(expected_answers, predicted_answers)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
factscore_score = factscore_metric(expected_answers, predicted_answers)
summac_zs_score = summac_zs_metric(expected_answers, predicted_answers)
summac_conv_score = summac_conv_metric(expected_answers, predicted_answers)

cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore_score, summac_zs_score, summac_conv_score

0it [00:00, ?it/s]

[2026-03-01 01:02:33.358] ---GENERATE STEP-BACK QUERY---
[2026-03-01 01:02:34.442] ---GENERATE SUBQUERIES---
[2026-03-01 01:02:34.879] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-01 01:02:35.350] ---SELECTED SOURCES: ['vectorstore']---
[2026-03-01 01:02:35.351] ---ROUTE QUESTION---
[2026-03-01 01:02:35.351] ---GENERATE HYDE DOCUMENTS---
[2026-03-01 01:02:36.693] ---RETRIEVE FROM VECTOR STORE---
[2026-03-01 01:02:38.476] ---GRADE DOCUMENTs---
[2026-03-01 01:02:38.476] ---AFTER EXACT DEDUPLICATION: 8 documents---
[2026-03-01 01:02:38.479] ---BM25 TOP CANDIDATES: 8 documents---
[2026-03-01 01:02:39.874] ---FINAL DOCUMENTS NUMBER: 0---
[2026-03-01 01:02:39.874] ---ASSESS GRADED DOCUMENTS---
[2026-03-01 01:02:39.874] ---DECISION: SOME DOCUMENTS ARE NOT RELEVANT TO QUESTION, INCLUDE WEB SEARCH---
[2026-03-01 01:02:39.875] ---WEB SEARCH---
[2026-03-01 01:02:41.759] ---GENERATE---


13it [00:19,  1.52s/it]


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 45979. To increase, visit https://openrouter.ai/settings/credits and add more credits', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_2vm4Dg9yqFCfvAd7p46zPaa8Yp3'}